# Heart Disease Prediction - Exploratory Data Analysis

This notebook performs initial dataset inspection, raw exploratory analysis, and target transformation documentation on the Heart Disease dataset acquired from the UCI Machine Learning Repository (Dataset ID: 45).

## 1. Dataset Source

- **Repository**: UCI Machine Learning Repository
- **Dataset Name**: Heart Disease Dataset (Cleveland subset)
- **Dataset ID**: 45
- **Official URL**: [https://archive.ics.uci.edu/dataset/45/heart+disease](https://archive.ics.uci.edu/dataset/45/heart+disease)
- **Acquisition**: Downloaded programmatically via `ucimlrepo`

## 2. Dataset Loading

We load the raw CSV file preserved at `data/raw/heart_disease_uci.csv` using a robust relative path.

In [ ]:
from pathlib import Path
import pandas as pd

# Resolve relative path to raw dataset
data_path = Path("../data/raw/heart_disease_uci.csv")
if not data_path.exists():
    data_path = Path("data/raw/heart_disease_uci.csv")

df = pd.read_csv(data_path)
print(f"Raw dataset loaded successfully from: {data_path.resolve()}")

## 3. Dataset Shape

In [ ]:
print(f"Number of Rows (Instances): {df.shape[0]}")
print(f"Number of Columns (Variables): {df.shape[1]}")

## 4. Feature Overview

In [ ]:
df.info()

## 5. First Records

In [ ]:
df.head()

## 6. Descriptive Statistics

In [ ]:
df.describe().T

## 7. Missing Value Assessment

In [ ]:
missing_counts = df.isnull().sum()
missing_df = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Missing_Percentage': (missing_counts / len(df)) * 100
})
print("Missing Value Summary:")
print(missing_df[missing_df['Missing_Count'] > 0])
print(f"\nTotal Missing Cells: {df.isnull().sum().sum()}")

## 8. Duplicate Assessment

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Number of Duplicate Rows: {duplicate_count}")

## 9. Target Variable Inspection

Inspection of raw target column `num` (0 = absence, 1-4 = presence/severity).

In [ ]:
target_counts = df['num'].value_counts().sort_index()
target_df = pd.DataFrame({
    'Raw_Target_Value': target_counts.index,
    'Instance_Count': target_counts.values,
    'Percentage': (target_counts.values / len(df)) * 100
})
print(target_df.to_string(index=False))

## 10. Binary Target for Machine Learning

For binary classification models, the original multiclass target `num` is mapped as follows:
- `0` -> **0** (No Heart Disease)
- `1, 2, 3, 4` -> **1** (Heart Disease Present)

### Feature Classification:
- **Continuous Numerical Features (5)**: `age`, `trestbps`, `chol`, `thalach`, `oldpeak`
- **Categorical / Discrete Features (8)**: `sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal`
  * *Note on `ca`*: `ca` represents number of major vessels colored by fluoroscopy (0-3). Treated as discrete categorical.

In [ ]:
y_binary = df['num'].apply(lambda x: 0 if x == 0 else 1)
binary_counts = y_binary.value_counts().sort_index()
binary_df = pd.DataFrame({
    'Binary_Target_Label': ['0 (No Heart Disease)', '1 (Heart Disease Present)'],
    'Instance_Count': binary_counts.values,
    'Percentage': (binary_counts.values / len(df)) * 100
})
print(binary_df.to_string(index=False))

## 11. Initial Observations

1. **Dataset Dimensions**: The dataset contains 303 patient records with 13 predictor features and 1 target attribute (`num`).
2. **Data Integrity**: Zero duplicate rows were found in the raw dataset.
3. **Missing Data**: Missing values are localized to two features: `ca` (4 missing values, 1.32%) and `thal` (2 missing values, 0.66%). Total missing cells equal 6.
4. **Target Structure**: Raw target `num` ranges from 0 to 4. Binary mapping yields 164 negative cases (`0`, 54.13%) and 139 positive cases (`1`, 45.87%).
5. **Preprocessing Strategy**: Preprocessing parameters (Median Imputation, Mode Imputation, Standard Scaling, One-Hot Encoding) are fitted strictly on the 80% training set to guarantee zero data leakage.